# Spatial dual-product identity — exact gate

This fail-closed notebook runs one immutable judge in normal and optimised Python. It licenses only the finite-product lift from the local dual-bond identity to `spatialKernel`; it does not license a weighted-ring, Jordan–Wigner, or spectral theorem.

In [ ]:
import datetime, hashlib, json, os, platform, subprocess, sys, tempfile, urllib.request
from pathlib import Path

RAW_COMMIT = '5ab0736b682583fb01644661ab72d30b4a455327'
RAW_URL = ('https://raw.githubusercontent.com/lluiseriksson/'
    f'THE-ERIKSSON-PROGRAMME/{RAW_COMMIT}/scripts/judge_spatial_dual_product.py')
EXPECTED_SHA256 = 'f975d229d0b4adc79844dfbc32afa440177f48cdc529fa8a1a8a418f2b5ed772'
EXPECTED_CHECKS = 5461
EXPECTED_MUTATIONS = 5460

run_root = Path(tempfile.mkdtemp(prefix='spatial-dual-product-'))
judge = run_root / 'judge_spatial_dual_product.py'
source = urllib.request.urlopen(RAW_URL, timeout=60).read()
source_hash = hashlib.sha256(source).hexdigest()
if source_hash != EXPECTED_SHA256:
    raise RuntimeError(f'judge SHA-256 mismatch: {source_hash}')
judge.write_bytes(source)

record = {
  'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'runtime': platform.platform(), 'python': platform.python_version(),
  'cpu_count': os.cpu_count(), 'raw_commit': RAW_COMMIT,
  'judge_sha256': source_hash, 'runs': [],
}
for flags in ([], ['-O']):
    command = [sys.executable, *flags, str(judge)]
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False)
    print('$', ' '.join(command))
    print(result.stdout, end='')
    print(f'[exit {result.returncode}]')
    if result.returncode != 0:
        raise RuntimeError(f'judge failed under flags {flags}')
    payload = json.loads(result.stdout)
    if payload.get('status') != 'PASS':
        raise RuntimeError(f'judge omitted PASS under flags {flags}')
    if payload.get('configuration_pairs_checked') != EXPECTED_CHECKS:
        raise RuntimeError(f'wrong check count under flags {flags}: {payload}')
    if payload.get('known_exponent_mutations_rejected') != EXPECTED_MUTATIONS:
        raise RuntimeError(f'wrong mutation count under flags {flags}: {payload}')
    record['runs'].append({'flags': flags, 'exit': result.returncode, 'payload': payload})

artifact = run_root / 'dual_product_gate.json'
artifact.write_text(json.dumps(record, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(f'artifact_sha256={hashlib.sha256(artifact.read_bytes()).hexdigest()}')
print('SPATIAL DUAL-PRODUCT GATE PASS')
from google.colab import files
files.download(str(artifact))